# 19 Gold Claim Cost Summary

## Purpose

This notebook creates a patient-level healthcare cost analytics table.

## What We Are Doing

We will analyze healthcare claim costs by patient, including:
- total claim cost
- average claim cost
- max claim cost
- pharmacy claim cost
- institutional claim cost
- claim volume
- high-cost patient flags

## Why We Are Doing This

Claim cost analytics is critical for:
- payer analytics
- provider financial analytics
- population health cost management
- high-cost patient identification
- Power BI financial dashboards
- ML cost prediction features

## Final Output

`healthcare_catalog.gold.claim_cost_summary`

## Final Grain

One row per patient.

## Step 1 — Import PySpark Functions

### What We Are Doing
We are importing Spark SQL functions.

### Why We Are Doing This
Cost analytics requires aggregation, conditional logic, and KPI calculations.

### Expected Output
Spark functions are available.

In [0]:
from pyspark.sql.functions import *

## Step 2 — Read Silver Claim Table

### What We Are Doing
We are reading the clean claim table from the Silver layer.

### Why We Are Doing This
Claim records contain healthcare billing and cost information.

### Expected Output
Claim DataFrame loaded successfully.

In [0]:
claim_df = spark.table(
    "healthcare_catalog.silver.claim_clean"
)

print("Claim Silver table loaded successfully.")

Claim Silver table loaded successfully.


## Step 3 — Standardize Claim Type

### What We Are Doing
We are extracting readable claim type codes from the FHIR claim type JSON text.

### Why We Are Doing This
The claim type is currently stored as JSON text.  
We need simple values such as:
- institutional
- pharmacy

### Expected Output
A clean column named `claim_type_code`.

In [0]:
claim_standardized_df = claim_df.withColumn(
    "claim_type_code",
    get_json_object(col("claim_type"), "$.coding[0].code")
)

display(claim_standardized_df)

claim_id,patient_reference,encounter_reference,claim_status,claim_use,claim_type,insurance_provider,billing_start,billing_end,total_claim_amount,claim_currency,payment_amount,diagnosis_reference,procedure_reference,patient_id,encounter_id,diagnosis_condition_id,procedure_id,claim_duration_days,claim_type_code
15f05361-dede-0d32-7e8b-455048db7f34,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,urn:uuid:a889805b-20ca-0ba6-5b5c-e617ce99b0cc,active,claim,"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""institutional""}]}",null,1954-04-03T17:14:47.000Z,1954-04-03T17:29:47.000Z,77.49,USD,null,null,null,b0a06ead-cc42-aa48-dad6-841d4aa679fa,a889805b-20ca-0ba6-5b5c-e617ce99b0cc,null,null,0,institutional
e435aaad-5d5c-bca6-7a3b-1aba157e01da,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,urn:uuid:b1ca5d4d-e3b5-e65b-e900-3844ae30f421,active,claim,"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""institutional""}]}",null,1954-04-20T13:14:47.000Z,1954-04-20T13:29:47.000Z,77.49,USD,null,null,null,b0a06ead-cc42-aa48-dad6-841d4aa679fa,b1ca5d4d-e3b5-e65b-e900-3844ae30f421,null,null,0,institutional
75c45f6c-bb5a-e92d-0cc2-af28a0516767,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,urn:uuid:853a4e65-da0d-d531-eb6e-458a70bcf552,active,claim,"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""institutional""}]}",null,1959-06-03T19:14:47.000Z,1959-06-03T19:33:08.000Z,624.9,USD,null,null,null,b0a06ead-cc42-aa48-dad6-841d4aa679fa,853a4e65-da0d-d531-eb6e-458a70bcf552,null,null,0,institutional
4244b2dc-9a09-b63e-1ad8-c7dd8a59be98,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,urn:uuid:b04db379-2092-c969-8968-39ab3557b605,active,claim,"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""institutional""}]}",null,1971-01-29T08:14:47.000Z,1971-01-29T08:29:47.000Z,1241.4199999999998,USD,null,urn:uuid:8695c578-79e7-b7fa-0d3b-be91b95d4a56,null,b0a06ead-cc42-aa48-dad6-841d4aa679fa,b04db379-2092-c969-8968-39ab3557b605,8695c578-79e7-b7fa-0d3b-be91b95d4a56,null,0,institutional
78d012c9-0b85-c1ba-abb7-ee76c10f19b5,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,urn:uuid:dffdcaf9-384a-3472-867c-2f852dc61851,active,claim,"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""institutional""}]}",null,1977-10-25T08:14:47.000Z,1977-10-25T08:29:47.000Z,5850.25,USD,null,null,null,b0a06ead-cc42-aa48-dad6-841d4aa679fa,dffdcaf9-384a-3472-867c-2f852dc61851,null,null,0,institutional
a4643f8b-0d6a-4f53-813f-b2497fe11aa9,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,urn:uuid:ed03c4f1-9865-7638-37d1-d7c1fe282363,active,claim,"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""institutional""}]}",null,1978-02-10T08:14:47.000Z,1978-02-10T08:29:47.000Z,786.3299999999999,USD,null,urn:uuid:c21907ce-d314-d9ac-87b5-e9dff5371c13,null,b0a06ead-cc42-aa48-dad6-841d4aa679fa,ed03c4f1-9865-7638-37d1-d7c1fe282363,c21907ce-d314-d9ac-87b5-e9dff5371c13,null,0,institutional
bfffde09-1959-12da-b6ed-cf5ee4827418,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,urn:uuid:e756ebcd-40d3-59f6-257c-4b413d4e94e8,active,claim,"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""institutional""}]}",null,1981-02-13T08:14:47.000Z,1981-02-13T08:29:47.000Z,1244.4199999999998,USD,null,urn:uuid:896c3120-7689-0630-2698-04d5a3f5f704,null,b0a06ead-cc42-aa48-dad6-841d4aa679fa,e756ebcd-40d3-59f6-257c-4b413d4e94e8,896c3120-7689-0630-2698-04d5a3f5f704,null,0,institutional
6ccba5c1-d15d-cb8f-94f1-07a8fedd2aab,urn:uuid:b0a06ead-cc42-aa48-dad6-841d4aa679fa,urn:uuid:e47502eb-e574-2058-f1f9-4f19844e4da8,active,claim,"{""coding"":[{""system"":""http://terminology.hl7.org/CodeSystem/claim-type"",""code"":""institutional""}]}",null,1982-04-22T22:14:47.000Z,1982-04-22T22:37:23.000Z,517.68,USD,null,null,null,b0a06ead-cc42-aa48-dad6-841d4aa679fa,e47502eb-e574-2058-f1f9-4f19844e4da8

## Step 4 — Create Patient-Level Claim Cost Summary

### What We Are Doing
We are aggregating claim cost metrics at the patient level.

### Why We Are Doing This
Gold cost tables should summarize financial utilization by patient.

### Metrics Created

- total claims
- total claim cost
- average claim cost
- maximum claim cost
- pharmacy claim count
- institutional claim count
- pharmacy claim cost
- institutional claim cost

### Expected Output
One row per patient with healthcare cost KPIs.

In [0]:
claim_cost_summary_df = claim_standardized_df.groupBy(
    "patient_id"
).agg(

    count("*").alias("total_claims"),

    sum("total_claim_amount").alias("total_claim_cost"),

    avg("total_claim_amount").alias("avg_claim_cost"),

    max("total_claim_amount").alias("max_claim_cost"),

    sum(
        when(col("claim_type_code") == "pharmacy", 1).otherwise(0)
    ).alias("pharmacy_claim_count"),

    sum(
        when(col("claim_type_code") == "institutional", 1).otherwise(0)
    ).alias("institutional_claim_count"),

    sum(
        when(
            col("claim_type_code") == "pharmacy",
            col("total_claim_amount")
        ).otherwise(0)
    ).alias("pharmacy_claim_cost"),

    sum(
        when(
            col("claim_type_code") == "institutional",
            col("total_claim_amount")
        ).otherwise(0)
    ).alias("institutional_claim_cost")
)

display(claim_cost_summary_df)

patient_id,total_claims,total_claim_cost,avg_claim_cost,max_claim_cost,pharmacy_claim_count,institutional_claim_count,pharmacy_claim_cost,institutional_claim_cost
b0a06ead-cc42-aa48-dad6-841d4aa679fa,85,120811.62000000001,1421.3131764705884,89974.97,53,32,4057.2399999999993,116754.38
ccfc4db2-2026-7adb-3db0-33f3828140bb,87,40134.75000000001,461.31896551724145,1967.13,49,38,10271.510000000002,29863.240000000016
31a2e8ec-69fc-8a71-3ab6-36cbdd508713,65,208955.26999999973,3214.6964615384572,64760.45,1,64,0.0,208955.26999999973
71a8b156-760b-df6b-859e-eefc7932a526,23,22147.81000000001,962.9482608695656,10551.46,4,19,130.12,22017.690000000006
76b289fd-e825-734c-8446-316f59643593,36,126532.93000000005,3514.8036111111123,60259.75000000001,12,24,104.03,126428.90000000005
92fb7efc-5cfd-f8d3-927b-42f8ee099531,34,15155.11,445.7385294117647,1448.57,7,27,70.5,15084.610000000002
81aa7647-779f-fd6b-94cf-782e606efeb2,18,10938.01,607.6672222222222,1177.28,1,17,5.96,10932.05
346a1435-2455-914f-c287-7b88052d05db,126,605520.1200000002,4805.71523809524,69511.58999999998,57,69,140.59999999999994,605379.5200000001
1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4,64,86857.63000000002,1357.1504687500003,18137.41,26,38,3203.8799999999997,83653.75000000001
97899f1d-9c3b-2b90-17b8-400c11ab8f0f,86,317028.87999999983,3686.382325581393,66527.89000000001,32,54,9455.199999999997,307573.6799999999


## Step 5 — Create Cost Risk Flags

### What We Are Doing
We are creating patient-level healthcare cost risk indicators.

### Why We Are Doing This
Healthcare organizations often identify high-cost patients for:
- case management
- care coordination
- payer analytics
- population health programs

### Flags Created

- high cost patient
- very high cost patient
- high pharmacy cost
- high institutional cost

### Expected Output
Cost risk flag columns.

In [0]:
claim_cost_summary_df = claim_cost_summary_df.withColumn(
    "high_cost_patient_flag",
    when(col("total_claim_cost") >= 100000, 1).otherwise(0)
).withColumn(
    "very_high_cost_patient_flag",
    when(col("total_claim_cost") >= 500000, 1).otherwise(0)
).withColumn(
    "high_pharmacy_cost_flag",
    when(col("pharmacy_claim_cost") >= 10000, 1).otherwise(0)
).withColumn(
    "high_institutional_cost_flag",
    when(col("institutional_claim_cost") >= 100000, 1).otherwise(0)
)

display(claim_cost_summary_df)

patient_id,total_claims,total_claim_cost,avg_claim_cost,max_claim_cost,pharmacy_claim_count,institutional_claim_count,pharmacy_claim_cost,institutional_claim_cost,high_cost_patient_flag,very_high_cost_patient_flag,high_pharmacy_cost_flag,high_institutional_cost_flag
b0a06ead-cc42-aa48-dad6-841d4aa679fa,85,120811.62000000001,1421.3131764705884,89974.97,53,32,4057.2399999999993,116754.38,1,0,0,1
ccfc4db2-2026-7adb-3db0-33f3828140bb,87,40134.75000000001,461.31896551724145,1967.13,49,38,10271.510000000002,29863.240000000016,0,0,1,0
31a2e8ec-69fc-8a71-3ab6-36cbdd508713,65,208955.26999999973,3214.6964615384572,64760.45,1,64,0.0,208955.26999999973,1,0,0,1
71a8b156-760b-df6b-859e-eefc7932a526,23,22147.81000000001,962.9482608695656,10551.46,4,19,130.12,22017.690000000006,0,0,0,0
76b289fd-e825-734c-8446-316f59643593,36,126532.93000000005,3514.8036111111123,60259.75000000001,12,24,104.03,126428.90000000005,1,0,0,1
92fb7efc-5cfd-f8d3-927b-42f8ee099531,34,15155.11,445.7385294117647,1448.57,7,27,70.5,15084.610000000002,0,0,0,0
81aa7647-779f-fd6b-94cf-782e606efeb2,18,10938.01,607.6672222222222,1177.28,1,17,5.96,10932.05,0,0,0,0
346a1435-2455-914f-c287-7b88052d05db,126,605520.1200000002,4805.71523809524,69511.58999999998,57,69,140.59999999999994,605379.5200000001,1,1,0,1
1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4,64,86857.63000000002,1357.1504687500003,18137.41,26,38,3203.8799999999997,83653.75000000001,0,0,0,0
97899f1d-9c3b-2b90-17b8-400c11ab8f0f,86,317028.87999999983,3686.382325581393,66527.89000000001,32,54,9455.199999999997,307573.6799999999,1,0,0,1


## Step 6 — Validate Cost Analytics KPIs

### What We Are Doing
We are calculating population-level cost KPIs.

### Why We Are Doing This
This validates Gold cost analytics and supports executive dashboard metrics.

### Expected Output
Healthcare financial analytics summary.

In [0]:
display(
    claim_cost_summary_df.select(

        avg("total_claims").alias("avg_claims_per_patient"),

        avg("total_claim_cost").alias("avg_total_claim_cost"),

        avg("avg_claim_cost").alias("population_avg_claim_cost"),

        avg("max_claim_cost").alias("avg_max_claim_cost"),

        avg("pharmacy_claim_cost").alias("avg_pharmacy_claim_cost"),

        avg("institutional_claim_cost").alias("avg_institutional_claim_cost"),

        sum("high_cost_patient_flag").alias("high_cost_patients"),

        sum("very_high_cost_patient_flag").alias("very_high_cost_patients"),

        sum("high_pharmacy_cost_flag").alias("high_pharmacy_cost_patients"),

        sum("high_institutional_cost_flag").alias("high_institutional_cost_patients")
    )
)

avg_claims_per_patient,avg_total_claim_cost,population_avg_claim_cost,avg_max_claim_cost,avg_pharmacy_claim_cost,avg_institutional_claim_cost,high_cost_patients,very_high_cost_patients,high_pharmacy_cost_patients,high_institutional_cost_patients
93.81621621621622,240605.0620180174,2552.02904473868,30736.386774774775,7360.597927927924,233244.46409009004,246,76,54,241


## Step 7 — Analyze Claim Type Distribution

### What We Are Doing
We are analyzing claim type counts.

### Why We Are Doing This
This helps validate the financial composition of the dataset.

### Expected Output
Claim type distribution table.

In [0]:
display(
    claim_standardized_df.groupBy(
        "claim_type_code"
    ).count().orderBy(
        desc("count")
    )
)

claim_type_code,count
institutional,27812
pharmacy,24256


## Step 8 — Save Gold Claim Cost Summary Table

### What We Are Doing
We are saving the patient-level claim cost summary into the Gold layer.

### Why We Are Doing This
This table becomes:
- Power BI cost analytics source
- payer analytics dataset
- financial risk feature store
- population health cost table

### Expected Output
A Delta table:

`healthcare_catalog.gold.claim_cost_summary`

In [0]:
claim_cost_summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare_catalog.gold.claim_cost_summary"
    )

print("Gold claim_cost_summary saved successfully.")

Gold claim_cost_summary saved successfully.


## Step 9 — Verify Gold Tables

### What We Are Doing
We are verifying Gold tables.

### Why We Are Doing This
We want to confirm successful Gold table creation.

### Expected Output
`claim_cost_summary` should appear in Gold schema.

In [0]:
spark.sql("""
SHOW TABLES IN healthcare_catalog.gold
""").show(truncate=False)

+--------+-------------------------------+-----------+
|database|tableName                      |isTemporary|
+--------+-------------------------------+-----------+
|gold    |chronic_disease_summary        |false      |
|gold    |claim_cost_summary             |false      |
|gold    |encounter_utilization_summary  |false      |
|gold    |medication_summary             |false      |
|gold    |observation_vitals_labs_summary|false      |
|gold    |patient_summary                |false      |
|gold    |procedure_careplan_summary     |false      |
+--------+-------------------------------+-----------+

